In this assignment, you will transition from static word embeddings to sequential modeling. You will reuse the IMDB dataset from Week 3 to build, train, and evaluate RNN and LSTM architectures.

Assignment Objectives:    
	1.	Sequential Learning: Implement padding and truncation to handle variable-length text

	2.  Architectural Construction: Manually 'wire' RNN and LSTM layers using Keras

	3.	The Vanishing Gradient: Conduct a controlled experiment varying sequence lengths to observe how RNNs 'forget' long-range context

	4. Performance Auditing: Compare sequential models against your Week 3 baseline using Accuracy, Precision, Recall, and F1-score.

Week 3 Reuse:
	•	Reuse tokenizer/vocab
	•	Reuse embeddings source (Word2Vec/GloVe)
	•	Reuse train/val/test splits

## Part 1: Preprocessing & GloVe Integration (10 Points)


In [ ]:
## Student Code Required ##
# Instruction: Initialize the Tokenizer and pad your sequences.
# You must also map your pre-trained vectors from Week 3
# into a weight matrix for the Keras Embedding layer.


# ==================== Part 1: Preprocessing ====================
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Define vocabulary and sequence constraints
VOCAB_SIZE = _____
MAX_LEN = _____

# Reuse the Week 3 tokenizer (do not fit a new tokenizer)
# Implement padding logic
def prepare_sequences(data, length):
    sequences = tokenizer.texts_to_sequences(data)
    return pad_sequences(sequences, maxlen=length, padding='_____')

# Map Week 3 vectors to the new vocabulary matrix
# Assumption: From Week 3, you already have access to pre-trained word embeddings
# in one of the following forms:
	##	embeddings_index[word] → np.array (e.g., GloVe dictionary), or
	##	word2vec_model.wv[word] → np.array (gensim Word2Vec).
  ## Use whichever representation you created in Week 3.

# embedding_matrix = np.zeros((VOCAB_SIZE, EMBEDDING_DIM))
# ... (Student implementation of mapping logic)

Exercise:
1.   Compare the raw text to its padded sequence. For an RNN, why is 'pre-padding' (adding zeros at the beginning) generally preferred over 'post-padding' (adding zeros at the end)?
2.   How does the choice of MAX_LEN impact the computational cost of the training phase?


## Part 2: RNN Implementation & Loss Curves (25 Points)

In [ ]:
## Student Code Required ##
# Instruction: Implement a SimpleRNN.
# You are responsible for selecting the hidden units
#  and dropout rates to prevent overfitting.

# ==================== Part 2: RNN Implementation ====================
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

# Define the sequential architecture
model_rnn = Sequential([
    Embedding(VOCAB_SIZE, Embedding_DIM=_____, weights=[embedding_matrix], trainable=False),
    SimpleRNN(_____),
    Dropout(_____),
    Dense(1, activation='sigmoid')
])

model_rnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# if labels are multi-class, replace sigmoid/binary_crossentropy with
# softmax/sparse_categorical_crossentropy

# Train the model and store the 'history' object for plotting
history_rnn = model_rnn.fit(X_train, y_train, epochs=5, validation_data= (X_val, y_val))
## use week 3 validation split

## Part 3: LSTM Implementation & Comparison(25 Points)

In [ ]:
## Student Code Required ##
# Instruction: Using the same pipeline, implement an LSTM.
# Observe how the gating mechanism affects training stability.

# ==================== Part 3: LSTM Implementation ====================
from tensorflow.keras.layers import LSTM

# Replicate the architecture using an LSTM layer
model_lstm = Sequential([
    Embedding(VOCAB_SIZE, 100, weights=[embedding_matrix], trainable=False),
    LSTM(_____), # Implement LSTM layer
    Dropout(_____),
    Dense(1, activation='sigmoid')
])

model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_lstm = model_lstm.fit(X_train, y_train, epochs=5, validation_data= (X_val, y_val))

Exercise:
1.   Analyze your accuracy and loss curves for both models. Did you observe 'exploding gradients' or a plateau in the SimpleRNN?
2.   Explain how the Forget Gate in the LSTM specifically addresses the issues you saw in the standard RNN



## Part 4: Hyperparameter Tuning & Vanishing Gradient(30 Points)


### Stop & Think: The Vanishing Gradient Hypothesis

**Before running the code below, verify your understanding of the theory.**

Recall from the lecture that SimpleRNNs have a "short-term memory" problem. Mathematically, this happens because the gradients (the signals used to update weights) become smaller and smaller as they travel back through time.

**Hypothesis Task:**
Predict what will happen when we force the SimpleRNN to remember a sequence of 200 words instead of just 100.

1.  **Stability**: Will the model continue to learn, or will the accuracy "flatline" early?
2.  **Bias** :If the model forgets the beginning of the review (where the context often is), what is its safest bet? Will it guess randomly, or will it just predict "Positive" for everything because most reviews are positive?


*Write your hypothesis here before proceeding.*

In [ ]:
## Student Code Required ##
# Instruction: Rerun your models using sequence lengths of 100 vs. 200
# Document the impact in a comparison table

# ==================== Part 4: Tuning Experiment ====================
# Student Task: Compare accuracy across varying sequence lengths
# Length 100 vs 200
# Hidden Units 64 vs 128
# Dropout 0.2

# Create a summary table (Pandas DataFrame) to document the results
  # Example: model_type, seq_len, hidden_units, val_acc, val_loss, notes

**Exercise:**

Diagnosing the "Memory Loss": Don't just look at the accuracy score—look at *how* the model failed.

**Task:**
Generate a **Confusion Matrix** for your `SimpleRNN` (Sequence Length = 200) on the validation set.

```python
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Generate predictions for the long-sequence RNN
# y_pred_rnn = ... (your prediction logic here)

# cm = confusion_matrix(y_val, y_pred_rnn)
# sns.heatmap(cm, annot=True, fmt='d')

**Questions:**

* The "Majority Vote" Trap: Look at your confusion matrix. Did the SimpleRNN actually "learn" the difference between positive and negative, or did it just predict the most frequent class for every review??

* The LSTM Difference: The LSTM introduces a "Cell State". Explain how this "highway" for information allows the LSTM to keep the gradient signal alive for 200 steps, whereas the SimpleRNN's standard hidden state lets it fade away.


## Part 5: Model Evaluation (10 Points)

In [ ]:
## Student Code Required ##
# Instruction: Evaluate your best-performing models on the test set only.
# Report classification report for both models (RNN and LSTM)
# Use the appropriate prediction strategy depending on whether the task is binary or multi-class

# ==================== Part 5: Final Evaluation ====================
from sklearn.metrics import classification_report

# Perform inference and generate the classification report
   # OPTION 1: Binary Classification
   # Use this if your labels are binary (e.g., 0/1)
   # y_pred = (model.predict(X_test) > 0.5).astype(int)

   # OPTION 2: Multi-class Classification
   # Use this if your labels have more than two classes
   # y_pred = np.argmax(model.predict(X_test), axis=1)


Exercise:

1.	Reviewing your Precision and Recall, does your model tend to misclassify negative reviews as positive, or vice versa?
2.	Given the results, would you recommend this model for a production-level sentiment analysis tool? Why or why not?